##  Bibliotecas:

In [4]:
from datapi import ApiClient
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import Point
from rasterio.transform import from_origin
import numpy as np
import os
from sqlalchemy import create_engine
import datetime
from datetime import timedelta, datetime
from pymongo import MongoClient
import json

## ERA5 Data

In [5]:
def download_variable(variable, year, month, day, times, latitude, longitude, dataset="reanalysis-era5-single-levels", formate="netcdf"):
    url = 'https://cds.climate.copernicus.eu/api'
    key = '61baae5c-60a9-44fe-af42-0f6984fe359e'

    client = ApiClient(
        url=(url),
        key=(key),
    )
    client.check_authentication()
    
    # Define a small area around the point
    area = [latitude + 0.25, longitude - 0.25, latitude - 0.25, longitude + 0.25]

    request = {
        "product_type": "reanalysis",
        "variable": [variable],
        "year": year,
        "month": month,
        "day": day,
        "time": times,
        "format": formate,
        "area": area,
    }
    filename = f"{variable}_{year}{month}{day}.nc"
    client.retrieve(dataset, request, filename)
    print(f"Downloaded {filename}")
    return filename
   
def process_variable(filename, variable_name):
    # Abrir o arquivo NetCDF
    ds = xr.open_dataset(filename)
   
    # Mapear os nomes das variáveis
    variable_mapping = {
        "10m_u_component_of_wind": "u10",
        "10m_v_component_of_wind": "v10",
        "2m_dewpoint_temperature": "d2m",
        "2m_temperature": "t2m",
        "surface_pressure": "sp",
        "total_precipitation": "tp",
        "maximum_2m_temperature_since_previous_post_processing": "mx2t",
        "minimum_2m_temperature_since_previous_post_processing": "mn2t",
        "total_column_rain_water": "tcrw",
        "surface_solar_radiation_downwards": "ssrd",
    }

    # Selecionar a variável desejada
    actual_variable_name = variable_mapping.get(variable_name, variable_name)
    variable = ds[actual_variable_name]

    data_list = []

    # Iterar sobre cada tempo disponível
    for time in variable.coords['valid_time']:
        # Selecionar os dados para o tempo atual
        data = variable.sel(valid_time=time)
        
        array = data.values
        lon = np.where(data.coords['longitude'].values > 180, data.coords['longitude'].values - 360, data.coords['longitude'].values)
        lat = data.coords['latitude'].values
        
        lon_grid, lat_grid = np.meshgrid(lon, lat)
        coords = np.array([(x, y) for x, y in zip(lon_grid.flatten(), lat_grid.flatten())])
        
        gdf = gpd.GeoDataFrame({
            'longitude': lon_grid.flatten(),
            'latitude': lat_grid.flatten(),
            'value': array.flatten(),
            'time': time.values
        }, geometry=[Point(x, y) for x, y in coords])
        
        gdf.set_crs(epsg=4326, inplace=True)
        
        data_list.append(gdf)

    # Concatenar todos os GeoDataFrames de dados
    combined_gdf = pd.concat(data_list, ignore_index=True)
    return combined_gdf

# Lista das variáveis que deseja baixar
variables = [
    "2m_temperature",
    "total_precipitation",
    "maximum_2m_temperature_since_previous_post_processing",
    "minimum_2m_temperature_since_previous_post_processing",
    "surface_solar_radiation_downwards",
]

latitude = -12.6542040849 
longitude = -45.6933894688

# Parâmetros da solicitação
today = datetime.today()
date = today - timedelta(days=6)
#date = datetime(2025, 2, 17)

year = date.strftime("%Y")
month = date.strftime("%m")
day = date.strftime("%d")
times = ["00:00", "01:00", "02:00", "03:00", "04:00", "05:00", "06:00", "07:00", "08:00", "09:00", "10:00", "11:00", "12:00", "13:00", "14:00", "15:00", "16:00", "17:00", "18:00", "19:00", "20:00", "21:00", "22:00", "23:00"]

# Baixar e processar cada variável
all_mean_gdfs = []
nc_files = []
for variable in variables:
    filename = download_variable(variable, year, month, day, times, latitude, longitude)
    if filename:
        mean_df = process_variable(filename, variable)
        mean_df['variable'] = variable  # Adicionar o nome da variável ao DataFrame
        all_mean_gdfs.append(mean_df)
        nc_files.append(filename)

# Concatenar todos os DataFrames de médias
if all_mean_gdfs:
    combined_mean_gdf = pd.concat(all_mean_gdfs, ignore_index=True)
    combined_mean_gdf = combined_mean_gdf.drop(columns=['longitude', 'latitude'])
    combined_mean_df = combined_mean_gdf.pivot_table(index=['time', 'geometry'], columns='variable', values='value').reset_index()
    
else:
    print("Nenhum dado foi processado.")

# Excluir os arquivos .nc
for nc_file in nc_files:
    os.remove(nc_file)
    print(f"Deleted {nc_file}")

[2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.
[2024-04-03T00:00:00] System is in degraded status due to underlaying infrastructure problems. Please follow status [here](https://status.ecmwf.int/)


Downloaded 2m_temperature_20250328.nc


[2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.
[2024-04-03T00:00:00] System is in degraded status due to underlaying infrastructure problems. Please follow status [here](https://status.ecmwf.int/)


Downloaded total_precipitation_20250328.nc


[2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.
[2024-04-03T00:00:00] System is in degraded status due to underlaying infrastructure problems. Please follow status [here](https://status.ecmwf.int/)


Downloaded maximum_2m_temperature_since_previous_post_processing_20250328.nc


[2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.
[2024-04-03T00:00:00] System is in degraded status due to underlaying infrastructure problems. Please follow status [here](https://status.ecmwf.int/)


Downloaded minimum_2m_temperature_since_previous_post_processing_20250328.nc


[2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.
[2024-04-03T00:00:00] System is in degraded status due to underlaying infrastructure problems. Please follow status [here](https://status.ecmwf.int/)


Downloaded surface_solar_radiation_downwards_20250328.nc
Deleted 2m_temperature_20250328.nc
Deleted total_precipitation_20250328.nc
Deleted maximum_2m_temperature_since_previous_post_processing_20250328.nc
Deleted minimum_2m_temperature_since_previous_post_processing_20250328.nc
Deleted surface_solar_radiation_downwards_20250328.nc


In [8]:
combined_mean_df

variable,time,geometry,2m_temperature,maximum_2m_temperature_since_previous_post_processing,minimum_2m_temperature_since_previous_post_processing,surface_solar_radiation_downwards,total_precipitation
0,2025-03-28 00:00:00,POINT (-45.944 -12.905),294.573792,296.593262,295.925201,0.0,0.000000e+00
1,2025-03-28 00:00:00,POINT (-45.6935 -12.905),295.319153,296.960388,296.435791,0.0,0.000000e+00
2,2025-03-28 00:00:00,POINT (-45.944 -12.654),294.922913,296.832092,296.275391,0.0,0.000000e+00
3,2025-03-28 00:00:00,POINT (-45.944 -12.404),294.734314,296.546448,296.063782,0.0,1.827539e-07
4,2025-03-28 00:00:00,POINT (-45.6935 -12.404),295.281952,296.878540,296.533447,0.0,0.000000e+00
...,...,...,...,...,...,...,...
211,2025-03-28 23:00:00,POINT (-45.6935 -12.404),295.759125,296.306396,295.696533,0.0,0.000000e+00
212,2025-03-28 23:00:00,POINT (-45.6935 -12.654),296.010223,296.538361,296.073425,0.0,0.000000e+00
213,2025-03-28 23:00:00,POINT (-45.443 -12.404),295.648010,296.149994,295.566650,0.0,0.000000e+00
214,2025-03-28 23:00:00,POINT (-45.443 -12.654),295.886932,296.292114,295.815948,0.0,0.000000e+00
